In [ ]:
# @title
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os

WORK_DIR = '/content/drive/MyDrive/logit'
os.chdir(WORK_DIR)

print('Current working directory:')
print(os.getcwd())

print('Files in this folder:')
print(os.listdir())

In [ ]:
!pip install -U transformers accelerate pandas openpyxl matplotlib tqdm scipy statsmodels

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_NAME = "Qwen/Qwen2.5-3B"
RUN_NAME = "qwen25_3b"

device = "cuda" if torch.cuda.is_available() else "cpu"

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    trust_remote_code=True
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    dtype=torch.float16 if device == "cuda" else torch.float32,
    device_map="auto" if device == "cuda" else None,
    trust_remote_code=True
)

if device == "cpu":
    model.to(device)

model.eval()

print("Loaded:", MODEL_NAME)
print("Device:", device)

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import os
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt

from tqdm import tqdm
from matplotlib.colors import TwoSlopeNorm, LinearSegmentedColormap

ROOT_DIR = "/content/drive/MyDrive/logit"
INPUT_XLSX = os.path.join(ROOT_DIR, "lexical_items.xlsx")

OUTPUT_DIR = os.path.join(ROOT_DIR, "3b")
os.makedirs(OUTPUT_DIR, exist_ok=True)

RUN_NAME = "qwen25_3b"

print("Root folder:", ROOT_DIR)
print("Output folder:", OUTPUT_DIR)
print("Root files:", os.listdir(ROOT_DIR))

df = pd.read_excel(INPUT_XLSX, sheet_name="lexical_items")
df = df.fillna("")

print("Dataset shape:", df.shape)
display(df.head())

In [ ]:
def get_candidate_first_token_ids(text):
    """
    Return possible first token IDs for a continuation string.
    Both raw and leading space variants are included.
    """
    text = str(text).strip()

    if text == "":
        return []

    variants = [text, " " + text]
    ids = set()

    for v in variants:
        token_ids = tokenizer.encode(v, add_special_tokens=False)
        if len(token_ids) > 0:
            ids.add(token_ids[0])

    return sorted(ids)


def decode_token_ids(ids):
    return [tokenizer.decode([int(i)]) for i in ids]


def get_token_count(text):
    text = str(text).strip()

    if text == "":
        return 0

    return len(tokenizer.encode(text, add_special_tokens=False))


def get_normed_hidden(hidden):
    """
    Apply final normalization before lm_head when available.
    This is needed for Qwen style models.
    """
    if hasattr(model, "model") and hasattr(model.model, "norm"):
        return model.model.norm(hidden)

    if hasattr(model, "transformer") and hasattr(model.transformer, "ln_f"):
        return model.transformer.ln_f(hidden)

    return hidden


def get_top_tokens(probs, k=10):
    values, indices = torch.topk(probs, k=k)
    out = []

    for val, idx in zip(values.detach().cpu().tolist(), indices.detach().cpu().tolist()):
        tok = tokenizer.decode([idx])
        out.append((int(idx), tok, float(val)))

    return out


def rank_from_logits(logits, token_id):
    """
    Rank 1 means highest logit.
    This avoids sorting the whole vocabulary explicitly.
    """
    token_logit = logits[int(token_id)]
    rank = int((logits > token_logit).sum().item() + 1)
    return rank

In [ ]:
token_rows = []

for _, row in df.iterrows():
    target_ids = get_candidate_first_token_ids(row["target_word"])

    approx_terms = [
        row["english_approx_1"],
        row["english_approx_2"],
        row["english_approx_3"],
        row["english_approx_4"]
    ]

    approx_ids = []
    approx_token_display = []
    approx_counts = []

    for term in approx_terms:
        term = str(term).strip()

        if term != "":
            ids = get_candidate_first_token_ids(term)
            approx_ids.extend(ids)

            approx_token_display.append(
                f"{term}: ids={ids}, tokens={decode_token_ids(ids)}"
            )

            approx_counts.append(get_token_count(term))

    approx_ids = sorted(set(approx_ids))

    if len(target_ids) == 0:
        flag = "FLAG_REVIEW"
        reason = "No target token ID found"

    elif len(approx_ids) == 0:
        flag = "FLAG_REVIEW"
        reason = "No English approximation token ID found"

    else:
        flag = "KEEP"
        reason = "Actual model tokenizer IDs available"

    token_rows.append({
        "item_id": row["item_id"],
        "language": row["language"],
        "category": row["category"],
        "target_word": row["target_word"],
        "target_token_ids": str(target_ids),
        "target_token_texts": str(decode_token_ids(target_ids)),
        "target_token_count": get_token_count(row["target_word"]),
        "english_approx_tokens": " | ".join(approx_token_display),
        "english_approx_first_token_ids": str(approx_ids),
        "max_english_approx_token_count": max(approx_counts) if len(approx_counts) > 0 else 0,
        "keep_or_flag": flag,
        "reason": reason
    })

actual_tok = pd.DataFrame(token_rows)

actual_tok_path = os.path.join(OUTPUT_DIR, f"actual_tokenization_check_{RUN_NAME}.xlsx")
actual_tok.to_excel(actual_tok_path, index=False)

print("Saved:", actual_tok_path)
display(actual_tok.head())
display(actual_tok["keep_or_flag"].value_counts())

# Robust statistics and metric sensitivity update

The old extraction only computed summed English approximation probability. The following cells replace the old layer extraction and item metric cells. They save sum, mean, and max English approximation metrics, then compute item level areas, confidence intervals, bootstrap contrasts, language specific contrasts, and fixed effect regression checks.


In [ ]:

# =========================
# NEW CELL 16
# Re-run logit lens extraction with sum / mean / max English approximation metrics
# This replaces the old Cell 16.
# =========================

import json

layer_rows = []

model.eval()

ENGLISH_APPROX_COLS = [
    "english_approx_1",
    "english_approx_2",
    "english_approx_3",
    "english_approx_4"
]

for _, row in tqdm(df.iterrows(), total=len(df)):
    item_id = row["item_id"]
    prompt = str(row["prompt"])

    target_ids = get_candidate_first_token_ids(row["target_word"])

    approx_ids = []
    approx_terms = []

    for col in ENGLISH_APPROX_COLS:
        term = str(row[col]).strip()

        if term != "":
            approx_terms.append(term)
            approx_ids.extend(get_candidate_first_token_ids(term))

    # Deduplicate English first-token IDs.
    # This is essential: otherwise the summed metric can be mechanically inflated.
    approx_ids = sorted(set(approx_ids))

    if len(target_ids) == 0 or len(approx_ids) == 0:
        continue

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        add_special_tokens=False
    ).to(model.device)

    with torch.inference_mode():
        outputs = model(
            **inputs,
            output_hidden_states=True
        )

    hidden_states = outputs.hidden_states

    for layer_idx, h in enumerate(hidden_states):
        h_last = h[:, -1, :]
        h_norm = get_normed_hidden(h_last)

        logits = model.lm_head(h_norm)[0]
        probs = torch.softmax(logits.float(), dim=-1)

        target_probs = probs[target_ids].detach().cpu().numpy()
        english_probs = probs[approx_ids].detach().cpu().numpy()

        p_target = float(np.sum(target_probs))

        p_english_sum = float(np.sum(english_probs))
        p_english_mean = float(np.mean(english_probs))
        p_english_max = float(np.max(english_probs))

        target_logits = logits[target_ids].detach().cpu()
        best_target_id = target_ids[int(torch.argmax(target_logits))]

        approx_logits = logits[approx_ids].detach().cpu()
        best_approx_id = approx_ids[int(torch.argmax(approx_logits))]

        rank_target = rank_from_logits(logits, best_target_id)
        rank_english_best = rank_from_logits(logits, best_approx_id)

        top10 = get_top_tokens(probs, k=10)

        row_out = {
            "item_id": item_id,
            "language": row["language"],
            "category": row["category"],
            "target_word": row["target_word"],
            "model_name": MODEL_NAME,
            "run_name": RUN_NAME,
            "layer": layer_idx,
            "p_target": p_target,

            # Legacy name for old plotting cells.
            # This remains the original summed English probability.
            "p_english": p_english_sum,

            # New sensitivity metrics.
            "p_english_sum": p_english_sum,
            "p_english_mean": p_english_mean,
            "p_english_max": p_english_max,

            "n_target_tokens": int(len(target_ids)),
            "n_english_tokens": int(len(approx_ids)),

            "rank_target": rank_target,
            "rank_english_best": rank_english_best,
            "target_token_ids": json.dumps([int(x) for x in target_ids]),
            "english_approx_token_ids": json.dumps([int(x) for x in approx_ids]),
            "english_approx_terms": json.dumps(approx_terms, ensure_ascii=False),
            "english_approx_probs": json.dumps([float(x) for x in english_probs])
        }

        for i, item in enumerate(top10, start=1):
            row_out[f"top_token_{i}_id"] = item[0]
            row_out[f"top_token_{i}"] = item[1]
            row_out[f"top_token_{i}_prob"] = item[2]

        layer_rows.append(row_out)

    del outputs
    del hidden_states

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

layerwise = pd.DataFrame(layer_rows)

layerwise_path = os.path.join(OUTPUT_DIR, f"layerwise_results_{RUN_NAME}.csv")
layerwise_sensitivity_path = os.path.join(OUTPUT_DIR, f"layerwise_results_{RUN_NAME}_sensitivity.csv")

layerwise.to_csv(layerwise_path, index=False)
layerwise.to_csv(layerwise_sensitivity_path, index=False)

print("Saved:", layerwise_path)
print("Saved:", layerwise_sensitivity_path)
print("Layerwise shape:", layerwise.shape)
display(layerwise.head())


In [ ]:
# =========================
# NEW CELL 17A
# Compute item level metrics for sum, mean, max
# This replaces the old Cell 17 item metric computation.
# =========================

import re

metric_specs = {
    "sum": "p_english_sum",
    "mean": "p_english_mean",
    "max": "p_english_max"
}

metric_rows = []

for item_id, group in layerwise.groupby("item_id"):
    group = group.sort_values("layer").reset_index(drop=True)
    first = group.iloc[0]
    final = group.iloc[-1]

    # Normalized layer coordinate for area calculation.
    # Raw area preserves your original definition.
    # Normalized area is safer for cross model comparisons.
    layer_x = group["layer"].to_numpy(dtype=float)
    if layer_x.max() > layer_x.min():
        layer_x_norm = (layer_x - layer_x.min()) / (layer_x.max() - layer_x.min())
    else:
        layer_x_norm = layer_x

    for metric_name, english_col in metric_specs.items():
        english_values = group[english_col].to_numpy(dtype=float)
        target_values = group["p_target"].to_numpy(dtype=float)

        english_peak = float(np.max(english_values))
        english_peak_layer = int(group.loc[np.argmax(english_values), "layer"])

        target_peak = float(np.max(target_values))
        target_peak_layer = int(group.loc[np.argmax(target_values), "layer"])

        dominance_values = np.maximum(english_values - target_values, 0)

        # Original style: sum across layers.
        english_dominance_area_raw = float(np.sum(dominance_values))

        # Normalized area: useful if models have different layer counts.
        english_dominance_area_norm = float(np.trapezoid(dominance_values, x=layer_x_norm))

        crossover_rows = group[group["p_target"] > group[english_col]]

        if len(crossover_rows) > 0:
            crossover_layer = int(crossover_rows.iloc[0]["layer"])
        else:
            crossover_layer = np.nan

        metric_rows.append({
            "item_id": item_id,
            "language": first["language"],
            "category": first["category"],
            "target_word": first["target_word"],
            "model_name": first["model_name"],
            "run_name": first["run_name"],
            "metric": metric_name,
            "english_probability_column": english_col,
            "n_english_tokens": int(first["n_english_tokens"]),
            "english_peak": english_peak,
            "english_peak_layer": english_peak_layer,
            "target_peak": target_peak,
            "target_peak_layer": target_peak_layer,
            "crossover_layer": crossover_layer,

            # Keep this column name compatible with your old paper tables.
            # It uses the original raw sum across layers definition.
            "english_dominance_area": english_dominance_area_raw,

            # Additional normalized version.
            "english_dominance_area_norm": english_dominance_area_norm,

            "final_p_target": float(final["p_target"]),
            "final_p_english": float(final[english_col]),
            "final_rank_target": int(final["rank_target"]),
            "final_rank_english_best": int(final["rank_english_best"])
        })

metrics_long = pd.DataFrame(metric_rows)

metrics_long_path = os.path.join(OUTPUT_DIR, f"metrics_by_item_long_{RUN_NAME}.csv")
metrics_long.to_csv(metrics_long_path, index=False)

# Legacy compatibility: old plotting cells use metrics.
# Here metrics is the original summed metric only.
metrics = metrics_long[metrics_long["metric"] == "sum"].copy()

metrics_path = os.path.join(OUTPUT_DIR, f"metrics_by_item_{RUN_NAME}.csv")
metrics_sum_path = os.path.join(OUTPUT_DIR, f"metrics_by_item_{RUN_NAME}_sum.csv")

metrics.to_csv(metrics_path, index=False)
metrics.to_csv(metrics_sum_path, index=False)

# Table 1: dataset and tokenization summary.
table1 = actual_tok.groupby(["language", "category"]).agg(
    n_items=("item_id", "count"),
    mean_target_token_count=("target_token_count", "mean"),
    sd_target_token_count=("target_token_count", "std"),
    mean_max_english_approx_token_count=("max_english_approx_token_count", "mean"),
    sd_max_english_approx_token_count=("max_english_approx_token_count", "std"),
    n_keep=("keep_or_flag", lambda x: (x == "KEEP").sum()),
    n_flag_review=("keep_or_flag", lambda x: (x == "FLAG_REVIEW").sum())
).reset_index()

# Table 2: legacy main metric table for summed metric.
table2 = metrics.groupby(["language", "category"]).agg(
    n_items=("item_id", "count"),
    mean_english_peak=("english_peak", "mean"),
    sd_english_peak=("english_peak", "std"),
    mean_english_dominance_area=("english_dominance_area", "mean"),
    sd_english_dominance_area=("english_dominance_area", "std"),
    mean_english_dominance_area_norm=("english_dominance_area_norm", "mean"),
    sd_english_dominance_area_norm=("english_dominance_area_norm", "std"),
    median_crossover_layer=("crossover_layer", "median"),
    mean_final_p_target=("final_p_target", "mean"),
    sd_final_p_target=("final_p_target", "std"),
    mean_final_rank_target=("final_rank_target", "mean"),
    sd_final_rank_target=("final_rank_target", "std")
).reset_index()

table1_csv = os.path.join(OUTPUT_DIR, f"table1_dataset_summary_{RUN_NAME}.csv")
table2_csv = os.path.join(OUTPUT_DIR, f"table2_main_metrics_{RUN_NAME}.csv")

table1_xlsx = os.path.join(OUTPUT_DIR, f"table1_dataset_summary_{RUN_NAME}.xlsx")
table2_xlsx = os.path.join(OUTPUT_DIR, f"table2_main_metrics_{RUN_NAME}.xlsx")

table1.to_csv(table1_csv, index=False)
table2.to_csv(table2_csv, index=False)

table1.to_excel(table1_xlsx, index=False)
table2.to_excel(table2_xlsx, index=False)

workbook_path = os.path.join(OUTPUT_DIR, f"results_workbook_{RUN_NAME}.xlsx")

# Excel cannot store some control characters that may appear when tokenizer
# outputs are decoded. Clean strings only for Excel export. CSV files above
# keep the original values.
ILLEGAL_EXCEL_CHARS = re.compile(r"[\x00-\x08\x0B\x0C\x0E-\x1F]")

def clean_for_excel_value(value):
    if isinstance(value, str):
        return ILLEGAL_EXCEL_CHARS.sub("", value)
    return value

def clean_df_for_excel(dataframe):
    cleaned = dataframe.copy()
    object_cols = cleaned.select_dtypes(include=["object"]).columns
    for col in object_cols:
        cleaned[col] = cleaned[col].map(clean_for_excel_value)
    return cleaned

with pd.ExcelWriter(workbook_path, engine="openpyxl") as writer:
    clean_df_for_excel(df).to_excel(writer, sheet_name="lexical_items", index=False)
    clean_df_for_excel(actual_tok).to_excel(writer, sheet_name="actual_tokenization", index=False)
    clean_df_for_excel(layerwise).to_excel(writer, sheet_name="layerwise_preview", index=False)
    clean_df_for_excel(metrics_long).to_excel(writer, sheet_name="metrics_long", index=False)
    clean_df_for_excel(metrics).to_excel(writer, sheet_name="metrics_sum", index=False)
    clean_df_for_excel(table1).to_excel(writer, sheet_name="table1_summary", index=False)
    clean_df_for_excel(table2).to_excel(writer, sheet_name="table2_sum_metrics", index=False)

print("Saved item level metrics and tables.")
print("Metrics long:", metrics_long_path)
print("Metrics sum:", metrics_path)
print("Table 1:", table1_csv)
print("Table 2:", table2_csv)
print("Workbook:", workbook_path)

display(table1)
display(table2)
display(metrics_long.head())

In [ ]:

# =========================
# NEW CELL 17B
# Descriptive statistics: mean, SD, and 95% CI by category and by language x category
# =========================

from scipy import stats

AREA_COL = "english_dominance_area"       # Original raw area.
# AREA_COL = "english_dominance_area_norm"  # Use this instead for cross-model comparisons.

def summarize_with_ci(group):
    values = group[AREA_COL].dropna().to_numpy(dtype=float)
    n = len(values)

    if n == 0:
        return pd.Series({
            "n": 0,
            "mean": np.nan,
            "sd": np.nan,
            "se": np.nan,
            "ci_low": np.nan,
            "ci_high": np.nan
        })

    mean = float(np.mean(values))
    sd = float(np.std(values, ddof=1)) if n > 1 else np.nan
    se = float(sd / np.sqrt(n)) if n > 1 else np.nan

    if n > 1:
        t_value = stats.t.ppf(0.975, df=n - 1)
        ci_low = float(mean - t_value * se)
        ci_high = float(mean + t_value * se)
    else:
        ci_low = np.nan
        ci_high = np.nan

    return pd.Series({
        "n": n,
        "mean": mean,
        "sd": sd,
        "se": se,
        "ci_low": ci_low,
        "ci_high": ci_high
    })

summary_by_category = (
    metrics_long
    .groupby(["run_name", "metric", "category"])
    .apply(summarize_with_ci)
    .reset_index()
)

summary_by_category_language = (
    metrics_long
    .groupby(["run_name", "metric", "language", "category"])
    .apply(summarize_with_ci)
    .reset_index()
)

summary_by_category_path = os.path.join(OUTPUT_DIR, f"summary_by_category_{RUN_NAME}.csv")
summary_by_category_language_path = os.path.join(OUTPUT_DIR, f"summary_by_category_language_{RUN_NAME}.csv")

summary_by_category.to_csv(summary_by_category_path, index=False)
summary_by_category_language.to_csv(summary_by_category_language_path, index=False)

print("Saved:", summary_by_category_path)
print("Saved:", summary_by_category_language_path)

display(summary_by_category)
display(summary_by_category_language)


In [ ]:
# =========================
# NEW CELL 17C FAST
# Bootstrap category differences for sum, mean, max
# Stratified by language
# =========================

import os
import numpy as np
import pandas as pd

# Use the item level long table from Cell 17A.
area_df = metrics_long.copy()

AREA_COL = "english_dominance_area"
N_BOOT = 10000
SEED = 42

print("Available categories:", sorted(area_df["category"].unique()))
print("Available metrics:", sorted(area_df["metric"].unique()))
print("Available languages:", sorted(area_df["language"].unique()))

def find_category(category_names, keyword):
    matches = [c for c in category_names if keyword.lower() in str(c).lower()]
    if len(matches) == 0:
        raise ValueError(f"No category found for keyword: {keyword}")
    if len(matches) > 1:
        print(f"Multiple matches for {keyword}: {matches}. Using {matches[0]}")
    return matches[0]

category_names = sorted(area_df["category"].unique())

abstract_cat = find_category(category_names, "abstract")
concrete_cat = find_category(category_names, "concrete")
culture_cat = find_category(category_names, "culture")

planned_contrasts = [
    (abstract_cat, concrete_cat),
    (culture_cat, concrete_cat),
    (abstract_cat, culture_cat)
]

def stratified_category_mean(data, category, area_col=AREA_COL):
    language_means = []

    for language in sorted(data["language"].unique()):
        values = data.loc[
            (data["language"] == language) &
            (data["category"] == category),
            area_col
        ].dropna().to_numpy(dtype=float)

        if len(values) > 0:
            language_means.append(np.mean(values))

    if len(language_means) == 0:
        return np.nan

    return float(np.mean(language_means))

def fast_stratified_bootstrap_difference(
    data,
    category_a,
    category_b,
    area_col=AREA_COL,
    n_boot=N_BOOT,
    seed=SEED
):
    rng = np.random.default_rng(seed)

    languages = sorted(data["language"].unique())

    observed = (
        stratified_category_mean(data, category_a, area_col) -
        stratified_category_mean(data, category_b, area_col)
    )

    boot_diffs = np.empty(n_boot, dtype=float)

    # Pre store arrays for speed.
    values_by_cell = {}

    for language in languages:
        for category in [category_a, category_b]:
            values = data.loc[
                (data["language"] == language) &
                (data["category"] == category),
                area_col
            ].dropna().to_numpy(dtype=float)

            values_by_cell[(language, category)] = values

    for i in range(n_boot):
        mean_a_by_language = []
        mean_b_by_language = []

        for language in languages:
            values_a = values_by_cell[(language, category_a)]
            values_b = values_by_cell[(language, category_b)]

            if len(values_a) > 0:
                sample_a = rng.choice(values_a, size=len(values_a), replace=True)
                mean_a_by_language.append(np.mean(sample_a))

            if len(values_b) > 0:
                sample_b = rng.choice(values_b, size=len(values_b), replace=True)
                mean_b_by_language.append(np.mean(sample_b))

        boot_diffs[i] = np.mean(mean_a_by_language) - np.mean(mean_b_by_language)

    ci_low, ci_high = np.percentile(boot_diffs, [2.5, 97.5])

    p_approx = 2 * min(
        np.mean(boot_diffs <= 0),
        np.mean(boot_diffs >= 0)
    )

    return {
        "contrast": f"{category_a} minus {category_b}",
        "category_a": category_a,
        "category_b": category_b,
        "estimate": observed,
        "ci_low": ci_low,
        "ci_high": ci_high,
        "p_approx": p_approx,
        "n_boot": n_boot
    }

bootstrap_rows = []

for model_name in sorted(area_df["model_name"].unique()):
    for run_name in sorted(area_df["run_name"].unique()):
        for metric_name in sorted(area_df["metric"].unique()):

            subset = area_df[
                (area_df["model_name"] == model_name) &
                (area_df["run_name"] == run_name) &
                (area_df["metric"] == metric_name)
            ].copy()

            if len(subset) == 0:
                continue

            print(f"Running bootstrap: model={model_name}, run={run_name}, metric={metric_name}")

            for category_a, category_b in planned_contrasts:
                result = fast_stratified_bootstrap_difference(
                    data=subset,
                    category_a=category_a,
                    category_b=category_b,
                    area_col=AREA_COL,
                    n_boot=N_BOOT,
                    seed=SEED
                )

                result["model_name"] = model_name
                result["run_name"] = run_name
                result["metric"] = metric_name
                result["area_column"] = AREA_COL

                bootstrap_rows.append(result)

bootstrap_results = pd.DataFrame(bootstrap_rows)

bootstrap_path = os.path.join(
    OUTPUT_DIR,
    f"bootstrap_category_differences_{RUN_NAME}.csv"
)

bootstrap_results.to_csv(bootstrap_path, index=False)

print("Saved bootstrap category differences:")
print(bootstrap_path)

display(bootstrap_results)

In [ ]:

# =========================
# NEW CELL 17D
# Language-specific bootstrap category differences
# This directly tests whether the pooled category pattern hides language heterogeneity.
# =========================

language_bootstrap_rows = []

for metric in METRIC_ORDER:
    for language in sorted(metrics_long["language"].unique()):
        subset = metrics_long[
            (metrics_long["metric"] == metric) &
            (metrics_long["language"] == language)
        ].copy()

        for category_a, category_b in planned_contrasts:
            values_a = subset[subset["category"] == category_a][AREA_COL].dropna().to_numpy(dtype=float)
            values_b = subset[subset["category"] == category_b][AREA_COL].dropna().to_numpy(dtype=float)

            if len(values_a) < 2 or len(values_b) < 2:
                continue

            rng = np.random.default_rng(42)
            boot_diffs = []

            for _ in range(10000):
                sample_a = rng.choice(values_a, size=len(values_a), replace=True)
                sample_b = rng.choice(values_b, size=len(values_b), replace=True)
                boot_diffs.append(float(np.mean(sample_a) - np.mean(sample_b)))

            boot_diffs = np.array(boot_diffs, dtype=float)
            ci_low, ci_high = np.percentile(boot_diffs, [2.5, 97.5])

            language_bootstrap_rows.append({
                "run_name": RUN_NAME,
                "metric": metric,
                "language": language,
                "area_col": AREA_COL,
                "contrast": f"{category_a} minus {category_b}",
                "estimate": float(np.mean(values_a) - np.mean(values_b)),
                "ci_low": float(ci_low),
                "ci_high": float(ci_high),
                "n_a": int(len(values_a)),
                "n_b": int(len(values_b))
            })

bootstrap_category_differences_by_language = pd.DataFrame(language_bootstrap_rows)

language_bootstrap_path = os.path.join(
    OUTPUT_DIR,
    f"bootstrap_category_differences_by_language_{RUN_NAME}.csv"
)

bootstrap_category_differences_by_language.to_csv(language_bootstrap_path, index=False)

print("Saved:", language_bootstrap_path)
display(bootstrap_category_differences_by_language)


In [ ]:

# =========================
# NEW CELL 17E
# Metric sensitivity and candidate-token-count diagnostics
# =========================

# 1. Does the category ordering stay the same under sum, mean, and max?
metric_sensitivity_summary = (
    metrics_long
    .groupby(["metric", "category"])
    .agg(
        n_items=("item_id", "count"),
        mean_area=(AREA_COL, "mean"),
        sd_area=(AREA_COL, "std"),
        mean_n_english_tokens=("n_english_tokens", "mean"),
        sd_n_english_tokens=("n_english_tokens", "std")
    )
    .reset_index()
)

metric_sensitivity_by_language = (
    metrics_long
    .groupby(["metric", "language", "category"])
    .agg(
        n_items=("item_id", "count"),
        mean_area=(AREA_COL, "mean"),
        sd_area=(AREA_COL, "std"),
        mean_n_english_tokens=("n_english_tokens", "mean"),
        sd_n_english_tokens=("n_english_tokens", "std")
    )
    .reset_index()
)

metric_sensitivity_path = os.path.join(
    OUTPUT_DIR,
    f"metric_sensitivity_summary_{RUN_NAME}.csv"
)

metric_sensitivity_language_path = os.path.join(
    OUTPUT_DIR,
    f"metric_sensitivity_by_language_{RUN_NAME}.csv"
)

metric_sensitivity_summary.to_csv(metric_sensitivity_path, index=False)
metric_sensitivity_by_language.to_csv(metric_sensitivity_language_path, index=False)

# 2. Candidate token count by category.
candidate_count_summary = (
    metrics_long[metrics_long["metric"] == "sum"]
    .groupby(["category"])
    .agg(
        n_items=("item_id", "count"),
        mean_n_english_tokens=("n_english_tokens", "mean"),
        sd_n_english_tokens=("n_english_tokens", "std"),
        min_n_english_tokens=("n_english_tokens", "min"),
        max_n_english_tokens=("n_english_tokens", "max")
    )
    .reset_index()
)

candidate_count_path = os.path.join(
    OUTPUT_DIR,
    f"english_candidate_count_by_category_{RUN_NAME}.csv"
)

candidate_count_summary.to_csv(candidate_count_path, index=False)

print("Saved:", metric_sensitivity_path)
print("Saved:", metric_sensitivity_language_path)
print("Saved:", candidate_count_path)

display(metric_sensitivity_summary)
display(metric_sensitivity_by_language)
display(candidate_count_summary)


In [ ]:

# =========================
# NEW CELL 17F
# Supplementary fixed-effect regression checks
# Use this as supporting evidence, not as the sole inferential basis.
# =========================

import statsmodels.formula.api as smf

regression_rows = []

for metric in METRIC_ORDER:
    subset = metrics_long[metrics_long["metric"] == metric].copy()

    if len(subset) == 0:
        continue

    # Main fixed-effect model: category + language.
    fit_main = smf.ols(
        f"{AREA_COL} ~ C(category) + C(language)",
        data=subset
    ).fit(cov_type="HC3")

    ci_main = fit_main.conf_int()

    for param in fit_main.params.index:
        regression_rows.append({
            "run_name": RUN_NAME,
            "metric": metric,
            "area_col": AREA_COL,
            "model": "category_plus_language",
            "term": param,
            "estimate": float(fit_main.params[param]),
            "std_error": float(fit_main.bse[param]),
            "ci_low": float(ci_main.loc[param, 0]),
            "ci_high": float(ci_main.loc[param, 1]),
            "p_value": float(fit_main.pvalues[param]),
            "n": int(fit_main.nobs),
            "r_squared": float(fit_main.rsquared)
        })

    # Interaction model: category * language.
    fit_interaction = smf.ols(
        f"{AREA_COL} ~ C(category) * C(language)",
        data=subset
    ).fit(cov_type="HC3")

    ci_interaction = fit_interaction.conf_int()

    for param in fit_interaction.params.index:
        regression_rows.append({
            "run_name": RUN_NAME,
            "metric": metric,
            "area_col": AREA_COL,
            "model": "category_by_language_interaction",
            "term": param,
            "estimate": float(fit_interaction.params[param]),
            "std_error": float(fit_interaction.bse[param]),
            "ci_low": float(ci_interaction.loc[param, 0]),
            "ci_high": float(ci_interaction.loc[param, 1]),
            "p_value": float(fit_interaction.pvalues[param]),
            "n": int(fit_interaction.nobs),
            "r_squared": float(fit_interaction.rsquared)
        })

# Candidate-count control only makes conceptual sense for the summed metric.
subset_sum = metrics_long[metrics_long["metric"] == "sum"].copy()

fit_count = smf.ols(
    f"{AREA_COL} ~ C(category) + C(language) + n_english_tokens",
    data=subset_sum
).fit(cov_type="HC3")

ci_count = fit_count.conf_int()

for param in fit_count.params.index:
    regression_rows.append({
        "run_name": RUN_NAME,
        "metric": "sum",
        "area_col": AREA_COL,
        "model": "category_plus_language_plus_candidate_count",
        "term": param,
        "estimate": float(fit_count.params[param]),
        "std_error": float(fit_count.bse[param]),
        "ci_low": float(ci_count.loc[param, 0]),
        "ci_high": float(ci_count.loc[param, 1]),
        "p_value": float(fit_count.pvalues[param]),
        "n": int(fit_count.nobs),
        "r_squared": float(fit_count.rsquared)
    })

regression_checks = pd.DataFrame(regression_rows)

regression_checks_path = os.path.join(
    OUTPUT_DIR,
    f"regression_checks_{RUN_NAME}.csv"
)

regression_checks.to_csv(regression_checks_path, index=False)

print("Saved:", regression_checks_path)
display(regression_checks)


In [ ]:
plt.rcParams.update({
    "font.size": 11,
    "axes.titlesize": 12,
    "axes.labelsize": 11,
    "legend.fontsize": 10,
    "xtick.labelsize": 10,
    "ytick.labelsize": 10,
    "figure.dpi": 120,
    "savefig.dpi": 300,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.edgecolor": "#666666",
    "axes.linewidth": 0.8,
    "grid.color": "#D9D9D9",
    "grid.linewidth": 0.6,
    "grid.alpha": 0.6
})

target_color = "#4C72B0"
english_color = "#DD8452"

category_order = ["concrete", "abstract", "culture_specific"]

category_titles = {
    "concrete": "Concrete",
    "abstract": "Abstract",
    "culture_specific": "Culture Specific"
}

fig, axes = plt.subplots(1, 3, figsize=(16, 4.8), sharey=True)

for ax, cat in zip(axes, category_order):
    sub = layerwise[layerwise["category"] == cat]

    mean_by_layer = sub.groupby("layer")[["p_target", "p_english"]].mean().reset_index()
    sd_by_layer = sub.groupby("layer")[["p_target", "p_english"]].std().reset_index().fillna(0)

    x = mean_by_layer["layer"].values

    y_target = mean_by_layer["p_target"].values
    y_english = mean_by_layer["p_english"].values

    sd_target = sd_by_layer["p_target"].values
    sd_english = sd_by_layer["p_english"].values

    ax.grid(True, axis="y")

    ax.plot(
        x,
        y_target,
        color=target_color,
        linewidth=2.2,
        label="Target language"
    )

    ax.fill_between(
        x,
        y_target - sd_target,
        y_target + sd_target,
        color=target_color,
        alpha=0.18
    )

    ax.plot(
        x,
        y_english,
        color=english_color,
        linewidth=2.2,
        label="English approximation"
    )

    ax.fill_between(
        x,
        y_english - sd_english,
        y_english + sd_english,
        color=english_color,
        alpha=0.18
    )

    ax.set_title(category_titles[cat], pad=10)
    ax.set_xlabel("Layer")
    ax.set_ylabel("Mean probability")
    ax.set_xlim(x.min(), x.max())
    ax.set_ylim(bottom=0)
    ax.legend(frameon=False, loc="best")

plt.tight_layout()

fig1_path = os.path.join(OUTPUT_DIR, f"figure1_layerwise_probabilities_{RUN_NAME}.png")
plt.savefig(fig1_path, bbox_inches="tight")
plt.show()

print("Saved:", fig1_path)

In [ ]:
pivot_data = layerwise.copy()
pivot_data["english_minus_target"] = pivot_data["p_english"] - pivot_data["p_target"]

ordered_items = metrics.sort_values(
    ["category", "language", "item_id"]
)["item_id"].tolist()

heat = pivot_data.pivot(
    index="item_id",
    columns="layer",
    values="english_minus_target"
).loc[ordered_items]

soft_diverging = LinearSegmentedColormap.from_list(
    "soft_diverging",
    ["#5B8CC0", "#F7F7F7", "#C97B63"]
)

vmax = np.nanmax(np.abs(heat.values))

if vmax == 0 or np.isnan(vmax):
    vmax = 1e-6

norm = TwoSlopeNorm(
    vmin=-vmax,
    vcenter=0,
    vmax=vmax
)

fig, ax = plt.subplots(figsize=(12, 10))

im = ax.imshow(
    heat.values,
    aspect="auto",
    cmap=soft_diverging,
    norm=norm
)

cbar = plt.colorbar(im, ax=ax, fraction=0.03, pad=0.02)
cbar.set_label("P(English approximation) minus P(target language)")

ax.set_xlabel("Layer")
ax.set_ylabel("Item")
ax.set_yticks(range(len(heat.index)))
ax.set_yticklabels(heat.index, fontsize=6)

ordered_metrics = metrics.set_index("item_id").loc[ordered_items]
cat_series = ordered_metrics["category"].tolist()

boundary_positions = []

for i in range(1, len(cat_series)):
    if cat_series[i] != cat_series[i - 1]:
        boundary_positions.append(i - 0.5)

for b in boundary_positions:
    ax.axhline(b, color="white", linewidth=1.2)

ax.set_title("Layerwise English Dominance Heatmap", pad=10)

plt.tight_layout()

fig3_path = os.path.join(OUTPUT_DIR, f"figure3_english_dominance_heatmap_{RUN_NAME}.png")
plt.savefig(fig3_path, bbox_inches="tight")
plt.show()

print("Saved:", fig3_path)

In [ ]:
culture_metrics = metrics[metrics["category"] == "culture_specific"].copy()

top_rows = []

for _, mrow in culture_metrics.iterrows():
    item_id = mrow["item_id"]
    peak_layer = int(mrow["english_peak_layer"])

    sub = layerwise[
        (layerwise["item_id"] == item_id) &
        (layerwise["layer"] == peak_layer)
    ]

    if len(sub) == 0:
        continue

    row = sub.iloc[0]

    out = {
        "item_id": item_id,
        "language": row["language"],
        "target_word": row["target_word"],
        "english_peak_layer": peak_layer,
        "p_target_at_peak_layer": row["p_target"],
        "p_english_at_peak_layer": row["p_english"],
        "rank_target_at_peak_layer": row["rank_target"],
        "rank_english_best_at_peak_layer": row["rank_english_best"]
    }

    for i in range(1, 11):
        out[f"top_token_{i}"] = row[f"top_token_{i}"]
        out[f"top_token_{i}_prob"] = row[f"top_token_{i}_prob"]

    top_rows.append(out)

culture_top_tokens = pd.DataFrame(top_rows)

culture_top_path = os.path.join(OUTPUT_DIR, f"culture_specific_top_tokens_{RUN_NAME}.xlsx")
culture_top_tokens.to_excel(culture_top_path, index=False)

print("Saved:", culture_top_path)
display(culture_top_tokens)

In [ ]:
# =========================
# PLOT SETUP CELL
# Load summary tables for plotting
# =========================

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# If these variables already exist in memory, this is harmless.
summary_category_path = os.path.join(OUTPUT_DIR, f"summary_by_category_{RUN_NAME}.csv")
summary_category_language_path = os.path.join(OUTPUT_DIR, f"summary_by_category_language_{RUN_NAME}.csv")
metrics_long_path = os.path.join(OUTPUT_DIR, f"metrics_by_item_long_{RUN_NAME}.csv")

summary_category = pd.read_csv(summary_category_path)
summary_category_language = pd.read_csv(summary_category_language_path)
metrics_long = pd.read_csv(metrics_long_path)

print(summary_category.head())
print(summary_category_language.head())
print(metrics_long.head())

def find_label(labels, keyword):
    labels = list(labels)
    matches = [x for x in labels if keyword.lower() in str(x).lower().replace("_", " ")]
    if len(matches) == 0:
        raise ValueError(f"No label found for keyword: {keyword}")
    return matches[0]

category_order = [
    find_label(summary_category["category"].unique(), "concrete"),
    find_label(summary_category["category"].unique(), "abstract"),
    find_label(summary_category["category"].unique(), "culture")
]

metric_order = ["sum", "mean", "max"]

# language labels may be de/fr/zh or longer names; this tries to find them robustly
language_values = list(summary_category_language["language"].unique())

def find_language_label(values, keyword_candidates):
    for candidate in keyword_candidates:
        matches = [x for x in values if candidate.lower() in str(x).lower()]
        if len(matches) > 0:
            return matches[0]
    raise ValueError(f"No language label found for: {keyword_candidates}")

language_order = [
    find_language_label(language_values, ["de", "german"]),
    find_language_label(language_values, ["fr", "french"]),
    find_language_label(language_values, ["zh", "chinese"])
]

print("Category order:", category_order)
print("Metric order:", metric_order)
print("Language order:", language_order)

In [ ]:
# =========================
# FIGURE 2
# Metric sensitivity across lexical categories
# =========================

fig, axes = plt.subplots(1, 3, figsize=(15, 5), sharey=True)

for ax, metric in zip(axes, metric_order):
    sub = summary_category[summary_category["metric"] == metric].copy()
    sub["category"] = pd.Categorical(sub["category"], categories=category_order, ordered=True)
    sub = sub.sort_values("category")

    x = np.arange(len(sub))
    y = sub["mean"].to_numpy()
    yerr_lower = y - sub["ci_low"].to_numpy()
    yerr_upper = sub["ci_high"].to_numpy() - y
    yerr = np.vstack([yerr_lower, yerr_upper])

    ax.bar(x, y)
    ax.errorbar(x, y, yerr=yerr, fmt="none", capsize=4)

    ax.set_xticks(x)
    ax.set_xticklabels(["Concrete", "Abstract", "Culture specific"], rotation=0)
    ax.set_title(metric.capitalize())
    ax.set_xlabel("Lexical category")
    ax.grid(axis="y", alpha=0.3)

axes[0].set_ylabel("Mean English dominance area")
fig.suptitle("Figure 2. Metric sensitivity of English dominance area across lexical categories", fontsize=16)
plt.tight_layout()

fig2_path_png = os.path.join(OUTPUT_DIR, f"figure2_metric_sensitivity_{RUN_NAME}.png")
fig2_path_pdf = os.path.join(OUTPUT_DIR, f"figure2_metric_sensitivity_{RUN_NAME}.pdf")

plt.savefig(fig2_path_png, dpi=300, bbox_inches="tight")
plt.savefig(fig2_path_pdf, bbox_inches="tight")
plt.show()

print("Saved:", fig2_path_png)
print("Saved:", fig2_path_pdf)

In [ ]:
# =========================
# FIGURE 3
# Language specific category patterns (sum metric only)
# =========================

main_metric = "sum"

fig, axes = plt.subplots(1, 3, figsize=(15, 5), sharey=True)

language_title_map = {
    language_order[0]: "German",
    language_order[1]: "French",
    language_order[2]: "Chinese"
}

for ax, language in zip(axes, language_order):
    sub = summary_category_language[
        (summary_category_language["metric"] == main_metric) &
        (summary_category_language["language"] == language)
    ].copy()

    sub["category"] = pd.Categorical(sub["category"], categories=category_order, ordered=True)
    sub = sub.sort_values("category")

    x = np.arange(len(sub))
    y = sub["mean"].to_numpy()
    yerr_lower = y - sub["ci_low"].to_numpy()
    yerr_upper = sub["ci_high"].to_numpy() - y
    yerr = np.vstack([yerr_lower, yerr_upper])

    ax.bar(x, y)
    ax.errorbar(x, y, yerr=yerr, fmt="none", capsize=4)

    ax.set_xticks(x)
    ax.set_xticklabels(["Concrete", "Abstract", "Culture specific"], rotation=0)
    ax.set_title(language_title_map[language])
    ax.set_xlabel("Lexical category")
    ax.grid(axis="y", alpha=0.3)

axes[0].set_ylabel("Mean English dominance area")
fig.suptitle("Figure 3. Language specific category patterns in English dominance area (sum metric)", fontsize=16)
plt.tight_layout()

fig3_path_png = os.path.join(OUTPUT_DIR, f"figure3_language_specific_sum_{RUN_NAME}.png")
fig3_path_pdf = os.path.join(OUTPUT_DIR, f"figure3_language_specific_sum_{RUN_NAME}.pdf")

plt.savefig(fig3_path_png, dpi=300, bbox_inches="tight")
plt.savefig(fig3_path_pdf, bbox_inches="tight")
plt.show()

print("Saved:", fig3_path_png)
print("Saved:", fig3_path_pdf)

In [ ]:
# =========================
# APPENDIX FIGURE
# Language x metric full grid
# =========================

fig, axes = plt.subplots(3, 3, figsize=(15, 12), sharey=True)

language_title_map = {
    language_order[0]: "German",
    language_order[1]: "French",
    language_order[2]: "Chinese"
}

for i, language in enumerate(language_order):
    for j, metric in enumerate(metric_order):
        ax = axes[i, j]

        sub = summary_category_language[
            (summary_category_language["metric"] == metric) &
            (summary_category_language["language"] == language)
        ].copy()

        sub["category"] = pd.Categorical(sub["category"], categories=category_order, ordered=True)
        sub = sub.sort_values("category")

        x = np.arange(len(sub))
        y = sub["mean"].to_numpy()
        yerr_lower = y - sub["ci_low"].to_numpy()
        yerr_upper = sub["ci_high"].to_numpy() - y
        yerr = np.vstack([yerr_lower, yerr_upper])

        ax.bar(x, y)
        ax.errorbar(x, y, yerr=yerr, fmt="none", capsize=4)
        ax.set_xticks(x)
        ax.set_xticklabels(["Concrete", "Abstract", "Culture specific"], rotation=20)
        ax.grid(axis="y", alpha=0.3)

        if i == 0:
            ax.set_title(metric.capitalize())
        if j == 0:
            ax.set_ylabel(language_title_map[language] + "\nMean EDA")

fig.suptitle("Appendix Figure. Category patterns by language and metric", fontsize=16)
plt.tight_layout()

appendix_fig_path_png = os.path.join(OUTPUT_DIR, f"appendix_language_metric_grid_{RUN_NAME}.png")
appendix_fig_path_pdf = os.path.join(OUTPUT_DIR, f"appendix_language_metric_grid_{RUN_NAME}.pdf")

plt.savefig(appendix_fig_path_png, dpi=300, bbox_inches="tight")
plt.savefig(appendix_fig_path_pdf, bbox_inches="tight")
plt.show()

print("Saved:", appendix_fig_path_png)
print("Saved:", appendix_fig_path_pdf)